# momentum-buffer-update — ex1: in-place momentum buffer update b = mu*b + g

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `momentum-buffer-update`. Running the final beacon cell reports progress against the `Optimizer: Momentum buffer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Momentum buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`momentum-buffer-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "momentum-buffer-update"
DD_SUBTOPIC = "Optimizer: Momentum buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Momentum buffer update `b = mu*b + g` — quick refresher

Classical momentum keeps a velocity buffer `b` that exponentially decays past gradients while accumulating new ones. The recurrence is:

```
b_t = mu * b_{t-1} + g_t      # update buffer
g_t = b_t                     # use buffer as the effective gradient
```

Two impl-level details that trip people up:

1. The buffer is updated IN PLACE: `b.copy_(self.mu * b + g)`. A rebind (`b = self.mu * b + g`) only rebinds the LOCAL variable; the entry in `self.b[i]` is unchanged, so next step uses the stale value.
2. The `g = b` assignment is by-reference. Later mutation of `g` would mutate the buffer — but we don't mutate `g` after this, so it's fine in the canonical impl.

### Exercise 1 — in-place momentum buffer update b = mu*b + g

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the classical momentum recurrence `b = mu*b + g` via `buffer.copy_()` so the optimizer's STATE list is mutated (not just the local variable rebound).
> Keywords: momentum, buffer-update, copy-inplace
> ```

**KCs targeted:** `momentum-recurrence-mu-b-plus-g`, `buffer-copy_-mutates-state-in-place`

Implement `ex1_momentum_step(buffer_list, grad_list, mu)`. Simulates one momentum-step pass over ALL params in an optimizer.

For each `(b, g)` pair drawn from `(buffer_list, grad_list)`:

1. Update IN PLACE: `b.copy_(mu * b + g)`. This mutates the tensor stored at `buffer_list[i]` — so next step sees the new value.
2. The effective gradient `g_eff` for this param is the new buffer value. Append `g_eff` to a list and return it.

**Critical:** the buffer must be mutated in place, not rebound. `buffer_list[i] = mu * buffer_list[i] + g` would also work IF you go through the index. But `b = mu * b + g` inside the loop rebinds the LOCAL `b` — the entry in `buffer_list` is untouched. Use `b.copy_(...)` to make the in-place semantics explicit.

Inputs:
- `buffer_list`: list of per-param buffers (mutated in place).
- `grad_list`: list of per-param gradients (NOT mutated).
- `mu`: float momentum coefficient.

Output: list of per-param effective gradients (the new buffer values).

The test runs TWO consecutive steps to catch the rebind bug — if the in-place mutation is missing, step 2 sees a zero buffer again instead of `mu*g1`.

In [ ]:
def ex1_momentum_step(buffer_list: list, grad_list: list, mu: float) -> list:
    """Update each buffer in place via b.copy_(mu*b + g). Return new buffer values."""
    raise NotImplementedError()


def _test_ex1():
    # Two params, two buffers (zero-init like a real optimizer at step 1).
    b1 = t.zeros(3)
    b2 = t.zeros(2, 2)
    buffers = [b1, b2]
    # Save the original tensors' identity so we can verify NO rebind happens.
    orig_b1_id = id(b1)
    orig_b2_id = id(b2)
    orig_b1_ptr = b1.data_ptr()

    # === Step 1 ===
    g1 = [t.tensor([1.0, 2.0, 3.0]), t.tensor([[0.5, -0.5], [1.0, -1.0]])]
    g_eff_1 = ex1_momentum_step(buffers, g1, mu=0.9)

    # At step 1 with zero-init buffer, b = mu*0 + g = g exactly.
    assert t.allclose(g_eff_1[0], g1[0]), (
        f'step 1: with zero buffer, g_eff should equal g; got {g_eff_1[0]} vs {g1[0]}'
    )
    assert t.allclose(g_eff_1[1], g1[1]), 'step 1: g_eff[1] mismatch'

    # Buffer must have been mutated IN PLACE: same object, same storage, new values.
    assert id(buffers[0]) == orig_b1_id, (
        'buffers[0] was REBOUND to a new tensor — '
        'use b.copy_(mu*b + g), not b = mu*b + g'
    )
    assert buffers[0].data_ptr() == orig_b1_ptr, 'buffers[0] storage was reallocated'
    assert t.allclose(buffers[0], g1[0]), (
        f'after step 1 buffers[0] should hold g1; got {buffers[0]}'
    )

    # === Step 2 (the rebind bug shows up here) ===
    g2 = [t.tensor([1.0, 1.0, 1.0]), t.zeros(2, 2)]
    g_eff_2 = ex1_momentum_step(buffers, g2, mu=0.9)
    # Expected: b_2 = 0.9 * g1 + g2 = 0.9*[1,2,3] + [1,1,1] = [1.9, 2.8, 3.7]
    expected_b1_after = t.tensor([1.9, 2.8, 3.7])
    assert t.allclose(g_eff_2[0], expected_b1_after), (
        f'step 2 g_eff[0]: got {g_eff_2[0]}, expected {expected_b1_after}; '
        f'if you see [1,1,1] then the buffer was not mutated at step 1 (rebind bug)'
    )
    assert t.allclose(buffers[0], expected_b1_after), (
        f'after step 2 buffers[0] should hold {expected_b1_after}, got {buffers[0]}'
    )
    # Same object still.
    assert id(buffers[0]) == orig_b1_id, 'buffers[0] rebound on step 2'

    # === Step 3 — verify mu=0 collapses to plain g ===
    b_fresh = t.zeros(4)
    g3 = [t.tensor([7.0, 8.0, 9.0, 10.0])]
    _ = ex1_momentum_step([b_fresh], g3, mu=0.0)
    assert t.allclose(b_fresh, g3[0]), 'mu=0: buffer should just equal g'

    # Input grad_list MUST NOT be mutated.
    g_in = t.tensor([5.0, 5.0])
    g_snap = g_in.clone()
    _ = ex1_momentum_step([t.zeros(2)], [g_in], mu=0.9)
    assert t.equal(g_in, g_snap), 'grad tensors must not be mutated'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_momentum_step(buffer_list, grad_list, mu):
    g_eff_list = []
    for b, g in zip(buffer_list, grad_list):
        b.copy_(mu * b + g)        # IN-PLACE — mutates buffer_list[i]
        g_eff_list.append(b)       # by-reference; the buffer IS the effective grad
    return g_eff_list
```

**Why `b.copy_(...)` and not `b = ...`.** Inside the for-loop, `b` is a LOCAL name bound to the tensor object at `buffer_list[i]`. `b = mu * b + g` rebinds the LOCAL name to a brand-new tensor; `buffer_list[i]` still points at the original zero tensor. Next iteration starts from zero again — the momentum never accumulates. This is the #1 bug in handwritten momentum/Adam impls.

**Equivalent forms.** `b.copy_(mu*b + g)`, `b.mul_(mu).add_(g)`, and `buffer_list[i] = mu * buffer_list[i] + g` all produce the right result. ARENA picks `copy_` because it makes the in-place semantics literal and matches the comment 'this does need to be inplace, since we're modifying the value in self.b'.

**Why we return `b`, not `mu*b + g`.** After `b.copy_(mu*b+g)`, the buffer IS the new value. Returning `b` by reference means downstream code that does `theta -= lr * g_eff` sees the updated buffer. Returning `mu*b+g` would compute the same value twice.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()